# Train CARL Ratio Estimator: POI + Nuisance Parameters

Train a classifier to estimate $r(x|\theta,\nu) = p(x|\theta,\nu) / p(x|\theta_{\text{SM}}, \nu=0)$
where both the physics parameters $\theta = (C_W/\Lambda^2, C_{\tilde{W}}/\Lambda^2)$
and nuisance parameters $\nu = (\nu_{\mu_R}, \nu_{\mu_F}, \nu_{\text{corr}})$ are inputs to the network.

This allows profiling over nuisance parameters during inference.

**Approach**: MadMiner's `sample_train_ratio` doesn't return the sampled nu values,
so we sample manually: draw random $(\theta, \nu)$ pairs, compute the morphed ratio
for each, and concatenate $[\theta, \nu]$ as the parameter input to the network.

**Input**: `data/lhe_data_semi_parametric_b1.h5`  
**Output**: Trained model saved to `models/carl_nuisance`

## 0. Setup

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from madminer.sampling import SampleAugmenter
from madminer.ml import ParameterizedRatioEstimator

logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.INFO,
)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

## 1. Inspect Setup

In [ ]:
sampler = SampleAugmenter("data/lhe_data_semi_parametric_b1.h5")

param_names = list(sampler.parameters.keys())
nuis_names = list(sampler.nuisance_parameters.keys())
n_params = len(param_names)
n_nuisance = len(nuis_names)

print(f"Physics params ({n_params}): {param_names}")
print(f"Nuisance params ({n_nuisance}): {nuis_names}")
print(f"Observables: {list(sampler.observables.keys())}")
print(f"Morphing: {sampler.morpher is not None}")
print(f"Nuisance morpher: {sampler.nuisance_morpher is not None}")

## 2. Extract Training Data with Random Nuisance Parameters

We call `sample_train_ratio` multiple times, each with a different random $\nu$ value.
For each call we record the $\nu$ used and append it to $\theta$ to form the
extended parameter vector $[\theta, \nu]$.

Nuisance parameters are conventionally centered at 0 with $\pm 1$ representing
$\pm 1\sigma$ variations.

In [ ]:
output_dir = "data/samples_carl_nuis"
Path(output_dir).mkdir(parents=True, exist_ok=True)

n_nu_points = 50       # number of distinct nu values
n_theta_per_nu = 20    # number of theta points per nu value
samples_per_call = 10_000

rng = np.random.default_rng(42)

all_x, all_theta_ext, all_y, all_r_xz = [], [], [], []

for i_nu in range(n_nu_points):
    # Draw a random nu point (Gaussian prior, sigma=1)
    nu_val = rng.normal(0, 1, size=n_nuisance)

    x, theta0, theta1, y, r_xz, t_xz, _ = sampler.sample_train_ratio(
        theta0=("random_morphing_points", (n_theta_per_nu, [
            ("flat", -20.0, 20.0),
            ("flat", -20.0, 20.0),
        ])),
        theta1=("benchmark", "sm"),
        nu0=("morphing_point", nu_val),
        nu1=None,  # denominator at nominal nu=0
        n_samples=samples_per_call,
        nuisance_score=False,
        test_split=0.0,
        validation_split=0.0,
        partition="all",
    )

    # Build extended theta: [theta_phys, nu] for numerator events,
    # [theta_phys, 0] for denominator events (y=1)
    nu_col = np.zeros((len(x), n_nuisance))
    num_mask = (y.flatten() == 0)
    nu_col[num_mask] = nu_val
    theta_ext = np.hstack([theta0, nu_col])

    all_x.append(x)
    all_theta_ext.append(theta_ext)
    all_y.append(y)
    all_r_xz.append(r_xz)

    if (i_nu + 1) % 10 == 0:
        print(f"  {i_nu + 1}/{n_nu_points} nu points done")

# Also add samples at nominal nu=0 (important for the network to learn the baseline)
for _ in range(10):
    x, theta0, theta1, y, r_xz, t_xz, _ = sampler.sample_train_ratio(
        theta0=("random_morphing_points", (n_theta_per_nu, [
            ("flat", -20.0, 20.0),
            ("flat", -20.0, 20.0),
        ])),
        theta1=("benchmark", "sm"),
        nu0=None,  # nominal
        nu1=None,
        n_samples=samples_per_call,
        nuisance_score=False,
        test_split=0.0,
        validation_split=0.0,
        partition="all",
    )
    nu_col = np.zeros((len(x), n_nuisance))
    theta_ext = np.hstack([theta0, nu_col])
    all_x.append(x)
    all_theta_ext.append(theta_ext)
    all_y.append(y)
    all_r_xz.append(r_xz)

# Concatenate and shuffle
X = np.vstack(all_x)
Theta = np.vstack(all_theta_ext)
Y = np.vstack(all_y)
R = np.vstack(all_r_xz)

perm = rng.permutation(len(X))
X, Theta, Y, R = X[perm], Theta[perm], Y[perm], R[perm]

# Train/test split
n_test = int(0.2 * len(X))
np.save(f"{output_dir}/x_train.npy", X[n_test:])
np.save(f"{output_dir}/theta_ext_train.npy", Theta[n_test:])
np.save(f"{output_dir}/y_train.npy", Y[n_test:])
np.save(f"{output_dir}/r_xz_train.npy", R[n_test:])
np.save(f"{output_dir}/x_test.npy", X[:n_test])
np.save(f"{output_dir}/theta_ext_test.npy", Theta[:n_test])
np.save(f"{output_dir}/y_test.npy", Y[:n_test])
np.save(f"{output_dir}/r_xz_test.npy", R[:n_test])

print(f"\nTotal samples: {len(X)}")
print(f"  Train: {len(X) - n_test}")
print(f"  Test:  {n_test}")
print(f"  Extended theta shape: {Theta.shape[1]} = {n_params} phys + {n_nuisance} nuis")

# Free memory
del all_x, all_theta_ext, all_y, all_r_xz, X, Theta, Y, R, x, theta0, theta1, y, r_xz, t_xz
del sampler
import gc; gc.collect()

## 3. Train CARL with Extended Parameters

The `ParameterizedRatioEstimator` treats whatever is passed as `theta` as the
parameter input to the network. By passing $[\theta_{\text{phys}}, \nu]$, the
network learns $\hat{r}(x | \theta, \nu)$ directly.

In [ ]:
estimator = ParameterizedRatioEstimator(
    n_hidden=(256, 256, 256),
    activation="tanh",
)

losses_train, losses_val = estimator.train(
    method="carl",
    x=f"{output_dir}/x_train.npy",
    y=f"{output_dir}/y_train.npy",
    theta=f"{output_dir}/theta_ext_train.npy",
    n_epochs=50,
    batch_size=256,
    initial_lr=1e-3,
    final_lr=1e-5,
    validation_split=0.25,
    early_stopping=True,
    early_stopping_patience=10,
)

estimator.save("models/carl_nuisance")
print("Model saved to models/carl_nuisance")

## 4. Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses_train, label="Train")
ax.plot(losses_val, label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("CARL (POI + Nuisance) Training")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 5. Evaluate: Predicted vs True

In [ ]:
model = ParameterizedRatioEstimator()
model.load("models/carl_nuisance")

x_test_arr = np.load(f"{output_dir}/x_test.npy")
theta_test_arr = np.load(f"{output_dir}/theta_ext_test.npy")
r_xz_test = np.load(f"{output_dir}/r_xz_test.npy")

batch_size = 10_000
log_r_parts = []
for start in range(0, len(x_test_arr), batch_size):
    end = min(start + batch_size, len(x_test_arr))
    lr, _ = model.evaluate_log_likelihood_ratio(
        x=x_test_arr[start:end],
        theta=theta_test_arr[start:end],
        test_all_combinations=False,
    )
    log_r_parts.append(lr)
log_r_hat = np.concatenate(log_r_parts)
log_r_true = np.log(r_xz_test.flatten())

print(f"Predicted: shape={log_r_hat.shape}, mean={log_r_hat.mean():.3f}")
print(f"True:      shape={log_r_true.shape}, mean={log_r_true.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
finite = np.isfinite(log_r_true) & np.isfinite(log_r_hat)
ax.scatter(log_r_true[finite], log_r_hat[finite], alpha=0.02, s=2, rasterized=True)
lims = [max(log_r_true[finite].min(), -20), min(log_r_true[finite].max(), 20)]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlabel(r"True $\log\, r$")
ax.set_ylabel(r"Predicted $\log\, \hat{r}$")
ax.set_title("CARL (POI+NP): Predicted vs True")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax = axes[1]
residuals = log_r_hat[finite] - log_r_true[finite]
ax.hist(residuals, bins=100, range=(-5, 5), density=True, alpha=0.7)
ax.axvline(0, color="r", ls="--")
ax.set_xlabel(r"$\log\,\hat{r} - \log\,r$")
ax.set_ylabel("Density")
ax.set_title(f"Residuals (mean={residuals.mean():.3f}, std={residuals.std():.3f})")

plt.tight_layout()
plt.show()

## 6. Nuisance Parameter Impact

Compare the log likelihood ratio at a fixed $\theta$ as a function of $\nu$
to verify the network has learned the nuisance dependence.

In [ ]:
# Pick a non-SM theta to see the effect
theta_fixed = np.array([10.0, 0.0])  # CWL2=10, CPWL2=0

# Vary one nuisance parameter at a time
x_sm = np.load(f"{output_dir}/x_test.npy")
y_test = np.load(f"{output_dir}/y_test.npy")
sm_mask = (y_test.flatten() == 1)
x_sm = x_sm[sm_mask][:500]
del y_test

nu_scan = np.linspace(-2, 2, 21)
nuis_labels = ["mu_R", "mu_F", "corr"]

fig, axes = plt.subplots(1, n_nuisance, figsize=(5 * n_nuisance, 4))
if n_nuisance == 1:
    axes = [axes]

for i_nuis in range(n_nuisance):
    mean_log_r = []
    for nu_val in nu_scan:
        nu_vec = np.zeros(n_nuisance)
        nu_vec[i_nuis] = nu_val
        theta_ext = np.hstack([theta_fixed, nu_vec])
        theta_batch = np.tile(theta_ext, (len(x_sm), 1))

        lr, _ = model.evaluate_log_likelihood_ratio(
            x=x_sm, theta=theta_batch, test_all_combinations=False,
        )
        mean_log_r.append(lr.mean())

    ax = axes[i_nuis]
    ax.plot(nu_scan, mean_log_r, "o-")
    ax.axvline(0, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(rf"$\nu_{{{nuis_labels[i_nuis]}}}$")
    ax.set_ylabel(r"$\langle \log\, \hat{r} \rangle_x$")
    ax.set_title(f"Impact of {nuis_labels[i_nuis]}")

plt.suptitle(rf"$\theta = ({theta_fixed[0]}, {theta_fixed[1]})$, varying one NP at a time", y=1.02)
plt.tight_layout()
plt.show()

## 7. Profiled NLL Scan

For each $\theta$ grid point, minimize $-2 \sum_i \log \hat{r}(x_i|\theta,\nu)$
over $\nu$ (profiling). Compare with the fixed $\nu=0$ result.

In [ ]:
from scipy.optimize import minimize

x_sm_prof = x_sm[:1000]

n_grid = 15
cwl2_vals = np.linspace(-15, 15, n_grid)
cpwl2_vals = np.linspace(-15, 15, n_grid)
cwl2_grid, cpwl2_grid = np.meshgrid(cwl2_vals, cpwl2_vals)

nll_fixed = np.zeros((n_grid, n_grid))
nll_profiled = np.zeros((n_grid, n_grid))
best_nu = np.zeros((n_grid, n_grid, n_nuisance))

def neg_log_lr(nu_vec, theta_phys, x_events):
    theta_ext = np.hstack([theta_phys, nu_vec])
    theta_batch = np.tile(theta_ext, (len(x_events), 1))
    lr, _ = model.evaluate_log_likelihood_ratio(
        x=x_events, theta=theta_batch, test_all_combinations=False,
    )
    return -2.0 * lr.sum()

for i in range(n_grid):
    for j in range(n_grid):
        theta_phys = np.array([cwl2_vals[j], cpwl2_vals[i]])

        # Fixed nu=0
        nll_fixed[i, j] = neg_log_lr(np.zeros(n_nuisance), theta_phys, x_sm_prof)

        # Profile over nu
        res = minimize(
            neg_log_lr, x0=np.zeros(n_nuisance),
            args=(theta_phys, x_sm_prof),
            method="L-BFGS-B",
            bounds=[(-3, 3)] * n_nuisance,
        )
        nll_profiled[i, j] = res.fun
        best_nu[i, j] = res.x

    print(f"  Row {i+1}/{n_grid} done")

nll_fixed -= nll_fixed.min()
nll_profiled -= nll_profiled.min()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, nll_data, title in [
    (axes[0], nll_fixed, r"Fixed $\nu = 0$"),
    (axes[1], nll_profiled, r"Profiled over $\nu$"),
]:
    c = ax.contourf(cwl2_grid, cpwl2_grid, nll_data, levels=30, cmap="viridis")
    plt.colorbar(c, ax=ax)
    ax.contour(cwl2_grid, cpwl2_grid, nll_data, levels=[2.30, 5.99],
               colors=["white", "lightgray"], linewidths=2)
    ax.plot(0, 0, "w*", markersize=15)
    ax.set_xlabel(r"$C_{W}/\Lambda^2$")
    ax.set_ylabel(r"$C_{\tilde{W}}/\Lambda^2$")
    ax.set_title(title)

plt.suptitle(r"$-2\Delta\log L$ comparison", fontsize=14)
plt.tight_layout()
plt.show()